# نوت‌بوک ۵ — مقاومت در برابر بازنویسی (ترجمه برگشتی)

**هدف:** متن‌های هوش مصنوعی در داده آزمون از مسیر فا→انگلیسی→فا ترجمه می‌شوند تا «امضای سبکی» مولد شکسته شود. سپس مدل baseline (بدون آموزش مجدد) روی این داده دستکاری‌شده ارزیابی می‌شود.

**متن‌های انسانی دست‌نخورده می‌مانند.** فقط ۶۰۰ متن AI بازنویسی می‌شود.

**مدت تخمینی:** حدود ۴۰ دقیقه روی T4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import time
import torch
import numpy as np

DRIVE_ROOT = Path('/content/drive/MyDrive/persian-ai-research')
ROBUST_DIR = DRIVE_ROOT / 'robustness'
ROBUST_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = DRIVE_ROOT / 'models' / 'baseline_model'

def load_jsonl(p):
    with open(p, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

test_records = load_jsonl(DRIVE_ROOT / 'data' / 'splits' / 'test.jsonl')
print(f'{len(test_records)} نمونه آزمون')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'دستگاه: {device}')

## بخش الف — بارگذاری مدل‌های ترجمه

نام دو مدل از parsinlu متفاوت است:
- فا→انگلیسی: `mt5-small-parsinlu-**opus**-translation_fa_en` (با opus)
- انگلیسی→فا: `mt5-small-parsinlu-translation_en_fa` (بدون opus)

In [ ]:
from transformers import MT5ForConditionalGeneration, MT5Tokenizer

FA_EN_MODEL = 'persiannlp/mt5-small-parsinlu-opus-translation_fa_en'
EN_FA_MODEL = 'persiannlp/mt5-small-parsinlu-translation_en_fa'

print('بارگذاری مدل فا→انگلیسی...')
tok_fa2en = MT5Tokenizer.from_pretrained(FA_EN_MODEL)
mdl_fa2en = MT5ForConditionalGeneration.from_pretrained(FA_EN_MODEL).to(device).eval()

print('بارگذاری مدل انگلیسی→فا...')
tok_en2fa = MT5Tokenizer.from_pretrained(EN_FA_MODEL)
mdl_en2fa = MT5ForConditionalGeneration.from_pretrained(EN_FA_MODEL).to(device).eval()

print('✓ آماده.')

In [ ]:
# توابع ترجمه
def translate(text, tokenizer, model, max_length=512):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_length=max_length, num_beams=2,
            early_stopping=True, no_repeat_ngram_size=3
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def back_translate(text):
    try:
        en = translate(text, tok_fa2en, mdl_fa2en)
        fa = translate(en, tok_en2fa, mdl_en2fa)
        return fa
    except Exception as e:
        print(f'  خطا: {e}')
        return None

## بخش ب — بازنویسی متن‌های AI (با قابلیت ادامه از وقفه)

اگر سشن قطع شود، با اجرای دوباره این سلول، از همان نقطه ادامه می‌دهد چون فایل خروجی پس از هر نمونه `flush` می‌شود.

In [ ]:
OUT_PATH = ROBUST_DIR / 'test_paraphrased.jsonl'

done_ids = set()
if OUT_PATH.exists():
    for line in open(OUT_PATH, encoding='utf-8'):
        if line.strip():
            done_ids.add(json.loads(line)['id'])
    print(f'{len(done_ids)} نمونه قبلاً بازنویسی شده.')

ai_test = [r for r in test_records if r['label'] == 1]
print(f'مجموع متن‌های AI: {len(ai_test)} | باقیمانده: {len(ai_test) - len(done_ids)}')

start = time.time()
count = 0
with open(OUT_PATH, 'a', encoding='utf-8') as fout:
    for r in ai_test:
        if r['id'] in done_ids:
            continue
        paraphrased = back_translate(r['text'])
        if not paraphrased:
            continue
        new_r = {**r, 'text': paraphrased, 'original_text': r['text'], 'paraphrased': True}
        fout.write(json.dumps(new_r, ensure_ascii=False) + '\n')
        fout.flush()
        count += 1
        if count % 20 == 0:
            elapsed = time.time() - start
            rate = count / elapsed
            remaining = (len(ai_test) - len(done_ids) - count) / rate / 60
            print(f'  {count} متن بازنویسی شد. باقیمانده تخمینی: {remaining:.1f} دقیقه')
print(f'\n✓ {count} متن جدید بازنویسی شد. کل: {len(done_ids) + count}')

## بخش ج — ارزیابی baseline روی داده دستکاری‌شده

متن‌های انسانی اصلی + متن‌های AI بازنویسی‌شده = داده آزمون جدید. مدل baseline بدون آموزش مجدد روی این داده پیش‌بینی می‌کند.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR)).to(device).eval()

# ساخت داده آزمون دستکاری‌شده
para_records = load_jsonl(OUT_PATH)
human_records = [r for r in test_records if r['label'] == 0]
modified_test = human_records + para_records
print(f'آزمون: {len(modified_test)} = {len(human_records)} انسان + {len(para_records)} AI بازنویسی‌شده')

def tokenize(b):
    return tokenizer(b['text'], truncation=True, max_length=256)
ds = Dataset.from_list(modified_test).map(tokenize, batched=True)

def metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1_macro': f1_score(labels, preds, average='macro')}

trainer = Trainer(model=model, tokenizer=tokenizer, data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=metrics)
pred_obj = trainer.predict(ds)
preds = np.argmax(pred_obj.predictions, axis=-1)
truth = pred_obj.label_ids

In [ ]:
# گزارش
overall_acc = (preds == truth).mean()
f1 = f1_score(truth, preds, average='macro')

human_idx = [i for i, r in enumerate(modified_test) if r['label'] == 0]
ai_idx = [i for i, r in enumerate(modified_test) if r['label'] == 1]
human_acc = (preds[human_idx] == truth[human_idx]).mean()
ai_acc = (preds[ai_idx] == truth[ai_idx]).mean()

print(f'دقت کلی: {overall_acc:.4f}')
print(f'F1 macro: {f1:.4f}')
print(f'دقت روی انسان (بدون بازنویسی): {human_acc:.4f}')
print(f'دقت روی AI (بازنویسی‌شده): {ai_acc:.4f}')

by_model = {}
for g in ['deepseek-v3.1', 'gpt-4o-mini', 'gemini-2.5-flash-lite']:
    idx = [i for i, r in enumerate(modified_test) if r.get('model') == g]
    a = (preds[idx] == truth[idx]).mean()
    by_model[g] = {'accuracy': float(a), 'count': len(idx)}
    print(f'  {g}: {a:.4f} ({int(a * len(idx))}/{len(idx)})')

out = {
    'experiment': 'robustness_paraphrasing',
    'baseline_accuracy': 0.9775,
    'paraphrased_accuracy': float(overall_acc),
    'f1_macro_paraphrased': float(f1),
    'accuracy_drop': 0.9775 - float(overall_acc),
    'human_accuracy': float(human_acc),
    'ai_accuracy_paraphrased': float(ai_acc),
    'ai_accuracy_drop': 0.99 - float(ai_acc),
    'by_model': by_model
}
with open(DRIVE_ROOT / 'results' / 'robustness_results.json', 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print('\n✓ ذخیره شد در results/robustness_results.json')